# Cumplimiento GDPR y EU AI Act en Sistemas de IA

> Aprende a evaluar y mejorar el cumplimiento normativo de sistemas de IA
> con GDPR y el EU AI Act usando Claude para auditorías automáticas.

## Objetivos
- Detectar PII innecesaria en prompts con Presidio
- Clasificar sistemas de IA por nivel de riesgo según el EU AI Act
- Auditar el cumplimiento GDPR de un sistema de IA
- Generar checklist de cumplimiento automatizado

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic presidio-analyzer --quiet
import subprocess
subprocess.run(["python", "-m", "spacy", "download", "es_core_news_sm"], check=False)

## 2. GDPR y IA: principios clave

In [ ]:
import anthropic
import json
import pandas as pd

client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

print("""=== GDPR Y SISTEMAS DE IA: PRINCIPIOS CLAVE ===

Art. 5 — Principios de tratamiento de datos:
  ✓ Minimización: solo los datos necesarios para el fin declarado
  ✓ Limitación del fin: no usar datos para fines distintos al original
  ✓ Exactitud: datos actualizados y correctos
  ✓ Limitación del plazo: no conservar más tiempo del necesario

Art. 22 — Decisiones automatizadas:
  ✓ Derecho a no ser objeto de decisiones basadas SOLO en tratamiento automatizado
  ✓ Excepciones: consentimiento explícito, contrato, ley
  ✓ Derecho a explicación y revisión humana

Art. 25 — Privacy by Design and by Default:
  ✓ Integrar privacidad desde el diseño del sistema
  ✓ Máxima privacidad por defecto

EU AI Act (2024) — Niveles de riesgo:
  🔴 Prohibido: vigilancia masiva, scoring social, manipulación subliminal
  🟠 Alto riesgo: RRHH, crédito, educación, justicia, infraestructuras críticas
  🟡 Riesgo limitado: chatbots, deepfakes (requieren transparencia)
  🟢 Riesgo mínimo: filtros de spam, recomendaciones de contenido
""")

## 3. Clasificar sistema de IA por nivel de riesgo (EU AI Act)

In [ ]:
def clasificar_riesgo_eu_ai_act(descripcion_sistema: str) -> dict:
    """Claude clasifica un sistema de IA según el EU AI Act."""
    prompt = f"""Eres un experto en el Reglamento de IA de la UE (EU AI Act 2024).
Clasifica el siguiente sistema de IA según su nivel de riesgo.

Sistema: {descripcion_sistema}

Niveles posibles:
- PROHIBIDO: sistemas que vulneran derechos fundamentales (scoring social, vigilancia masiva, manipulación)
- ALTO: empleo/RRHH, crédito, educación, sistemas críticos, justicia, migración
- LIMITADO: chatbots, sistemas de IA que generan contenido (requieren transparencia)
- MINIMO: filtros spam, recomendaciones, videojuegos con IA

Responde SOLO con JSON:
{{"nivel_riesgo": "PROHIBIDO|ALTO|LIMITADO|MINIMO",
  "justificacion": "<razón concreta citando el artículo>",
  "obligaciones_principales": ["obligación1", "obligación2"],
  "puede_operar": true/false}}"""

    resp = client.messages.create(
        model=MODELO, max_tokens=400,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text.strip()

    if "```" in resp:
        resp = resp.split("```")[1].lstrip("json").strip()
    return json.loads(resp)

SISTEMAS = [
    "Chatbot de atención al cliente para una tienda online de ropa",
    "Sistema de puntuación de CVs para selección de personal en una empresa",
    "Herramienta de reconocimiento facial en tiempo real para vigilancia de espacios públicos",
    "Filtro automático de spam en el correo electrónico"
]

ICONOS = {"PROHIBIDO": "🔴", "ALTO": "🟠", "LIMITADO": "🟡", "MINIMO": "🟢"}

print("=== CLASIFICACIÓN EU AI ACT ===")
for sistema in SISTEMAS:
    resultado = clasificar_riesgo_eu_ai_act(sistema)
    icono = ICONOS.get(resultado["nivel_riesgo"], "❓")
    print(f"\n{icono} {resultado['nivel_riesgo']}: '{sistema[:60]}'")
    print(f"   {resultado['justificacion'][:120]}")
    if not resultado["puede_operar"]:
        print("   ⛔ NO PUEDE OPERAR en la UE")

## 4. Auditoría GDPR automatizada

In [ ]:
def auditar_gdpr(descripcion_sistema: str, datos_procesados: list[str],
                 base_legal: str) -> dict:
    """Audita el cumplimiento GDPR de un sistema de IA con Claude."""
    prompt = f"""Eres un experto en GDPR. Audita el cumplimiento de este sistema de IA.

Sistema: {descripcion_sistema}
Datos personales procesados: {', '.join(datos_procesados)}
Base legal declarada: {base_legal}

Evalúa el cumplimiento de los principios del Art. 5 GDPR y el Art. 22 (decisiones automatizadas).

Responde con JSON:
{{
  "puntuacion_cumplimiento": 0-10,
  "principios_cumplidos": ["principio1"],
  "principios_incumplidos": ["principio1"],
  "riesgos_identificados": ["riesgo1"],
  "acciones_requeridas": ["accion1"],
  "requiere_dpia": true/false
}}"""

    resp = client.messages.create(
        model=MODELO, max_tokens=600,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text.strip()
    if "```" in resp:
        resp = resp.split("```")[1].lstrip("json").strip()
    return json.loads(resp)

auditoria = auditar_gdpr(
    descripcion_sistema="Sistema de scoring de crédito que usa ML para decidir automáticamente si aprobar préstamos",
    datos_procesados=["nombre completo", "DNI", "historial bancario", "ingresos", "dirección", "edad"],
    base_legal="Interés legítimo del responsable del tratamiento"
)

print("=== AUDITORÍA GDPR ===")
print(f"Puntuación de cumplimiento: {auditoria['puntuacion_cumplimiento']}/10")
print(f"Requiere DPIA: {'✓ SÍ' if auditoria['requiere_dpia'] else '✗ No'}")
print(f"\nPrincipios cumplidos: {', '.join(auditoria['principios_cumplidos'])}")
print(f"Principios incumplidos: {', '.join(auditoria['principios_incumplidos'])}")
print("\nRiesgos identificados:")
for r in auditoria["riesgos_identificados"]:
    print(f"  ⚠ {r}")
print("\nAcciones requeridas:")
for a in auditoria["acciones_requeridas"]:
    print(f"  → {a}")

## 5. Generar checklist de cumplimiento

In [ ]:
CHECKLIST_GDPR_IA = [
    ("Base legal documentada (Art. 6)", auditoria["puntuacion_cumplimiento"] >= 5),
    ("Minimización de datos (Art. 5.1.c)", "datos innecesarios" not in str(auditoria["riesgos_identificados"]).lower()),
    ("Información a interesados (Art. 13/14)", None),  # Requiere verificación manual
    ("Derecho de oposición implementado (Art. 21)", None),
    ("Decisiones automatizadas: revisión humana (Art. 22)", auditoria["puntuacion_cumplimiento"] >= 7),
    ("DPIA realizada si necesario (Art. 35)", not auditoria["requiere_dpia"]),
    ("Medidas de seguridad técnicas (Art. 32)", None),
    ("Registro de actividades de tratamiento (Art. 30)", None),
    ("DPO designado si obligatorio (Art. 37)", None),
]

print("=== CHECKLIST DE CUMPLIMIENTO GDPR + EU AI ACT ===")
total = len(CHECKLIST_GDPR_IA)
superados = 0
for item, superado in CHECKLIST_GDPR_IA:
    if superado is True:
        estado = "✅"
        superados += 1
    elif superado is False:
        estado = "❌"
    else:
        estado = "⚠️ "
    print(f"  {estado} {item}")

print(f"\nPuntuación checklist: {superados}/{total} ítems confirmados")

# Exportar checklist
checklist_md = "# Checklist de Cumplimiento GDPR + EU AI Act\n\n"
for item, superado in CHECKLIST_GDPR_IA:
    marca = "x" if superado else " "
    checklist_md += f"- [{marca}] {item}\n"

with open("checklist_cumplimiento.md", "w", encoding="utf-8") as f:
    f.write(checklist_md)
print("\n✓ Checklist exportado a 'checklist_cumplimiento.md'")